# LIME for tabular classification

This tutorial explains one prediction from a small scikit-learn classifier with Explainiverse's `LimeExplainer`. It is deterministic, runs on CPU, and uses scikit-learn's built-in Iris dataset, so it needs no network access or downloaded data.

The goal is to demonstrate the current API and its output contract. It is not an explanation-quality benchmark. LIME fits a weighted linear surrogate around one instance; its coefficients describe that local surrogate under LIME's sampling, kernel, and discretization choices.

Method reference: Ribeiro, Singh, and Guestrin (2016), *Why Should I Trust You? Explaining the Predictions of Any Classifier*, KDD, DOI 10.1145/2939672.2939778.

## Setup

Run this notebook from an installed Explainiverse checkout. It uses only declared runtime dependencies: NumPy, pandas, scikit-learn, and `lime`. No installation cell is needed. A single seed controls the data split, model, and LIME perturbation sampler.

In [1]:
import numpy as np
import pandas as pd
import sklearn

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from explainiverse.adapters.sklearn_adapter import SklearnAdapter
from explainiverse.core.explanation import Explanation
from explainiverse.evaluation import compute_pgi, compute_pgu
from explainiverse.explainers.attribution.lime_wrapper import LimeExplainer

SEED = 42
np.random.seed(SEED)
print(f"scikit-learn {sklearn.__version__}; fixed seed {SEED}")

scikit-learn 1.7.2; fixed seed 42


## Load data and train a model

Iris has 150 rows, four continuous measurements, and three target classes. Continuous numeric features fit the capabilities exposed by the current wrapper. The stratified split and bounded random forest keep this example quick and repeatable.

In [2]:
iris = load_iris()
X = np.asarray(iris.data, dtype=np.float64)
y = np.asarray(iris.target, dtype=np.int64)
feature_names = [str(name) for name in iris.feature_names]
class_names = [str(name) for name in iris.target_names]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

model = RandomForestClassifier(
    n_estimators=60,
    max_depth=3,
    random_state=SEED,
    n_jobs=1,
)
model.fit(X_train, y_train)
adapter = SklearnAdapter(
    model,
    feature_names=feature_names,
    class_names=class_names,
    task="classification",
)

test_probabilities = adapter.predict(X_test)
test_accuracy = float(model.score(X_test, y_test))

assert X_train.shape[1] == len(feature_names) == 4
assert class_names == ["setosa", "versicolor", "virginica"]
assert np.array_equal(model.classes_, np.arange(len(class_names)))
assert test_probabilities.shape == (len(X_test), len(class_names))
assert np.all(np.isfinite(test_probabilities))
assert np.all((0.0 <= test_probabilities) & (test_probabilities <= 1.0))
np.testing.assert_allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-12)
assert test_accuracy >= 0.75
print(f"train rows={len(X_train)}, test rows={len(X_test)}, test accuracy={test_accuracy:.3f}")

train rows=112, test rows=38, test accuracy=0.947


## Explain one explicit output

For classification, `target_class` is an **output-column index**, not a display label. We first inspect the adapter's probability vector, select its largest-probability column, and pass that integer explicitly.

`top_labels=1` is required by the current single-explanation return type. It does not mean class index 1. To explain another class, change `target_class` while leaving `top_labels=1`. `num_features=None` requests a complete vector over the four input features.

In [3]:
instance = np.asarray(X_test[0], dtype=np.float64)
instance_probabilities = adapter.predict(instance.reshape(1, -1))[0]
target_index = int(np.argmax(instance_probabilities))

lime_explainer = LimeExplainer(
    model=adapter,
    training_data=X_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    random_state=SEED,
)
explanation = lime_explainer.explain(
    instance,
    num_features=None,
    top_labels=1,
    target_class=target_index,
)

print("probabilities:", dict(zip(class_names, instance_probabilities.round(4))))
print(f"explained output column: {target_index} ({class_names[target_index]})")

probabilities: {'setosa': np.float64(1.0), 'versicolor': np.float64(0.0), 'virginica': np.float64(0.0)}
explained output column: 0 (setosa)


## Validate and inspect the result

Explainiverse maps LIME's feature-index weights back to the original feature names. A positive coefficient pushes the local surrogate toward the selected output; a negative coefficient pushes it away, within the sampled neighborhood. Magnitude is local-surrogate magnitude, not a causal effect or global feature importance.

In [4]:
assert isinstance(explanation, Explanation)
assert explanation.explainer_name == "LIME"
assert explanation.target_class == class_names[target_index]
assert explanation.feature_names == feature_names

attributions = explanation.get_attributions()
assert attributions is not None
assert list(attributions) == feature_names
attribution_values = np.asarray([attributions[name] for name in feature_names], dtype=float)
assert attribution_values.shape == (len(feature_names),)
assert np.all(np.isfinite(attribution_values))

details = explanation.explanation_data
serialized_prediction = np.asarray(details["model_prediction"], dtype=float)
np.testing.assert_allclose(serialized_prediction, instance_probabilities, atol=1e-12)
assert np.isfinite(float(details["local_prediction"]))
assert np.isfinite(float(details["local_model_score"]))

attribution_table = pd.DataFrame(
    {"feature": feature_names, "lime_coefficient": attribution_values}
).assign(abs_coefficient=lambda frame: frame["lime_coefficient"].abs())
attribution_table = attribution_table.sort_values(
    "abs_coefficient", ascending=False, ignore_index=True
)
print(attribution_table.to_string(index=False))
print(f"local surrogate score: {float(details['local_model_score']):.3f}")

          feature  lime_coefficient  abs_coefficient
 petal width (cm)          0.433918         0.433918
petal length (cm)          0.367523         0.367523
sepal length (cm)          0.057130         0.057130
 sepal width (cm)         -0.002339         0.002339
local surrogate score: 0.739


The optional `lime_feature_conditions` field contains the human-readable, possibly discretized conditions produced by the upstream LIME object. The canonical `feature_attributions` mapping above remains keyed by the four original columns.

In [5]:
for condition, weight in details["lime_feature_conditions"]:
    print(f"{weight:+.4f}  {condition}")

+0.4339  petal width (cm) <= 0.30
+0.3675  petal length (cm) <= 1.58
+0.0571  sepal length (cm) <= 5.10
-0.0023  3.00 < sepal width (cm) <= 3.32


## Check seeded repeatability

The upstream LIME sampler advances its random-number state after a call. Therefore, repeatability is checked with a **fresh explainer** configured with the same seed, data, model, instance, and target—not by calling the same stateful object twice.

In [6]:
repeat_explainer = LimeExplainer(
    model=adapter,
    training_data=X_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    random_state=SEED,
)
repeat_explanation = repeat_explainer.explain(
    instance,
    num_features=None,
    top_labels=1,
    target_class=target_index,
)
repeat_attributions = repeat_explanation.get_attributions()
assert repeat_attributions is not None
repeat_values = np.asarray([repeat_attributions[name] for name in feature_names], dtype=float)
np.testing.assert_allclose(repeat_values, attribution_values, rtol=0.0, atol=1e-12)
print("Fresh explainers with identical seeds produced identical coefficients.")

Fresh explainers with identical seeds produced identical coefficients.


## Optional perturbation diagnostics

Explainiverse's `compute_pgi` and `compute_pgu` below are deterministic baseline-replacement diagnostics. With `k=2`, PGI replaces the two highest-absolute-attribution features; PGU replaces their complement. The mean baseline is computed from the explicitly supplied training background.

These values measure output sensitivity to this particular intervention. They are not, by themselves, proof that LIME is faithful or causal, and they are not numerically interchangeable with noisy-expectation or area-under-curve benchmark variants from other libraries.

In [7]:
pgi = compute_pgi(
    adapter,
    instance,
    explanation,
    k=2,
    baseline="mean",
    background_data=X_train,
    target_class=target_index,
)
pgu = compute_pgu(
    adapter,
    instance,
    explanation,
    k=2,
    baseline="mean",
    background_data=X_train,
    target_class=target_index,
)
assert np.isfinite(pgi) and pgi >= 0.0
assert np.isfinite(pgu) and pgu >= 0.0
print(f"PGI={pgi:.4f}; PGU={pgu:.4f}")

PGI=0.9000; PGU=0.1152


## What this example does—and does not—establish

The assertions verify the software contract: probability shape, explicit output identity, complete finite attribution data, serialized prediction consistency, and seeded repeatability across fresh explainers. Model accuracy checks only that the tiny demonstration model learned a useful decision rule; it does not validate an explanation.

LIME explanations can change with the background data, seed, neighborhood sampling, kernel, discretization, and surrogate fit. The current Explainiverse tabular wrapper does not expose every upstream LIME option (for example, categorical-feature metadata, kernel width, or sample count), so this numeric Iris example should not be copied unchanged to mixed-type production data. For consequential use, define the target output explicitly, audit preprocessing and the perturbation distribution, inspect local surrogate quality, repeat across seeds, and evaluate with domain-appropriate tests.